In [1]:
import requests
import re
from bs4 import BeautifulSoup
from io import BytesIO
import zipfile

In [2]:
%run ../Framework/ADLSClient.ipynb

In [3]:
def download_and_upload(url, adls_container_client, adls_directory, latest_downloaded_file):
    try:
        print(f"Retrieving zip file from {url}.")
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        zip_file = BytesIO(response.content)
        print(f"{url} has been successfully stored in memory.")

        with zipfile.ZipFile(zip_file) as zf:
            for file_name in zf.namelist():
                if file_name.endswith('/'):
                    continue

                with zf.open(file_name) as file_data:
                    upload_path = f"{adls_directory}/{url.split('/')[-2]}/{file_name}"
                    print(f"Uploading {file_name} to ADLS at {upload_path}.")
                    
                    blob_client = adls_container_client.get_blob_client(upload_path)
                    blob_client.upload_blob(file_data.read(), overwrite=True)
                    print(f"Uploaded {file_name} successfully.")
    
    except requests.exceptions.RequestException as e:
        print(f"Failed to download ZIP file: {e}")
        
    except zipfile.BadZipFile:
        print("The downloaded file is not a valid ZIP file.")
        
    except Exception as e:
        print(f"An error occurred: {e}")

In [4]:
# create ADLS container client
adls_client = ADLSClient()
adls_container_client = adls_client.create_container_client()
adls_dir = "fhwa_import"

# get all downloadable content on the FHWA website
url = "https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/"
html = requests.get(url).content
soup = BeautifulSoup(html, "html.parser")

# all links to downloadable zip file
links = [a["href"] for a in soup.find_all("a", href=True)]
links = [f"{url}{link}" for link in links if ".zip" in link]

# get latest downloaded file in ADLS to treat as an offset
latest_ingested_file = max(
    ['/'.join(blob.name.split('/')[-2:]) for blob in adls_container_client.list_blobs() 
    if "fhwa_import/" in blob.name 
    and "Delimited_CleanData" not in blob.name]
)

# filter for any files greater than the lastest offset
urls = [link for link in links 
        if re.findall(r"/(\d{4})/", link)[0] >= re.findall(r"(\d{4})", latest_ingested_file)[0]]

In [6]:
for url in urls:
    download_and_upload(url, adls_container_client, adls_dir, latest_ingested_file)

Retrieving zip file from https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/2019/2019_station_data.zip.
https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/2019/2019_station_data.zip has been successfully stored in memory.
Uploading Station_Data_Extract_Pipe_Delimited_CleanData_2019.txt to ADLS at fhwa_import/2019/Station_Data_Extract_Pipe_Delimited_CleanData_2019.txt.
Uploaded Station_Data_Extract_Pipe_Delimited_CleanData_2019.txt successfully.
Retrieving zip file from https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/2019/jan_2019_ccs_data.zip.
https://www.fhwa.dot.gov/policyinformation/tables/tmasdata/2019/jan_2019_ccs_data.zip has been successfully stored in memory.
Uploading january_2019/AK0119.VOL to ADLS at fhwa_import/2019/january_2019/AK0119.VOL.
Uploaded january_2019/AK0119.VOL successfully.
Uploading january_2019/AL0119.VOL to ADLS at fhwa_import/2019/january_2019/AL0119.VOL.
Uploaded january_2019/AL0119.VOL successfully.
Uploading january_2019/AR0119